<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 16


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс PaymentMethod в C#, который будет представлять
различные способы оплаты. На основе этого класса разработать 2-3 производных
класса, демонстрирующих принципы наследования и полиморфизма. В каждом из
классов должны быть реализованы новые атрибуты и методы, а также
переопределены некоторые методы базового класса для демонстрации
полиморфизма.
Требования к базовому классу PaymentMethod:
• Атрибуты: ID способа оплаты (PaymentMethodId), Название способа оплаты
(MethodName), Минимальная сумма (MinAmount).
• Методы:
o ProcessPayment(decimal amount): метод для обработки платежа
указанной суммы.
o CheckMinimumAmount(decimal amount): метод для проверки
минимальной суммы платежа.
o GetPaymentDetails(): метод для получения деталей способа оплаты.
Требования к производным классам:
1. ОнлайнОплата (OnlinePayment): Должен содержать дополнительные
атрибуты, такие как URL платежной системы (PaymentUrl).
Метод ProcessPayment() должен быть переопределен для включения URL
платежной системы в процесс оплаты.
2. БанковскийПеревод (BankTransfer): Должен содержать дополнительные
атрибуты, такие как Банковские данные (BankData).
Метод CheckMinimumAmount() должен быть переопределен для проверки
минимальной суммы платежа с учетом банковских комиссий.
3. Наличные (CashPayment) (если требуется третий класс): Должен содержать
дополнительные атрибуты, такие как Место выдачи наличных
(CashPickupPoint). Метод GetPaymentDetails() должен быть переопределен
для отображения места выдачи наличных.

#### Дополнительное задание
Добавьте к сущестующим классам конструктора классов с использованием гетторов и сетторов и реализуйте взаимодействие объектов между собой

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте простое, сложное и множественное наследование

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [6]:
using System;
using System.Collections.Generic;

// ИНТЕРФЕЙСЫ
public interface IPaymentMethod
{
    int PaymentMethodId { get; set; }
    string MethodName { get; set; }
    decimal MinAmount { get; set; }

    bool CheckMinimumAmount(decimal amount);
    string ProcessPayment(decimal amount);
    string GetPaymentDetails();

    string GetSummary();
}

public interface IRefundable
{
    string Refund(decimal amount);
}

public interface ILoyaltyProgram
{
    int PointsEarned { get; set; }
    void AddPoints(decimal amount);
}

public interface ISecure
{
    bool IsEncrypted { get; set; }
    string Encrypt();
}

// БАЗОВЫЙ АБСТРАКТНЫЙ КЛАСС
public abstract class PaymentMethod : IPaymentMethod
{
    private decimal _minAmount;
    private bool _isActive = true;

    public int PaymentMethodId { get; set; }
    public string MethodName { get; set; }
    public string Currency { get; set; } = "RUB";
    public string Description { get; set; }

    public decimal MinAmount
    {
        get => _minAmount;
        set
        {
            if (value < 0) throw new ArgumentException("MinAmount не может быть отрицательным.");
            _minAmount = value;
        }
    }

    public bool IsActive
    {
        get => _isActive;
        set => _isActive = value;
    }

    public IPaymentMethod FallbackMethod { get; set; }

    protected PaymentMethod(int id, string methodName, decimal minAmount, string description = "")
    {
        PaymentMethodId = id;
        MethodName = methodName;
        MinAmount = minAmount;
        Description = description;
    }

    public virtual bool CheckMinimumAmount(decimal amount) => amount >= MinAmount;

    public virtual string ProcessPayment(decimal amount)
    {
        if (!IsActive)
            return $"Метод '{MethodName}' отключен.";

        if (!CheckMinimumAmount(amount))
        {
            if (FallbackMethod != null && FallbackMethod != this)
                return $"Сумма {amount:C} < минимума ({MinAmount:C}) для '{MethodName}'. " +
                       $"Резерв:\n   -> {FallbackMethod.ProcessPayment(amount)}";
            return $"Ошибка: сумма {amount:C} меньше минимальной ({MinAmount:C}).";
        }
        return $"Оплата {amount:C} ({Currency}) через '{MethodName}' выполнена.";
    }

    public virtual string GetPaymentDetails()
        => $"[ID {PaymentMethodId}] {MethodName} | Мин: {MinAmount:C} | Валюта: {Currency} | " +
           $"Активен: {IsActive} | {Description}";

    public void Activate() => IsActive = true;
    public void Deactivate() => IsActive = false;

    public virtual string GetSummary()
        => $"{MethodName} (ID={PaymentMethodId}, min={MinAmount})";
}

// ПРОСТОЕ НАСЛЕДОВАНИЕ + МНОЖЕСТВЕННОЕ (через интерфейсы)

public class OnlinePayment : PaymentMethod, ISecure, ILoyaltyProgram, IRefundable
{
    private string _paymentUrl = string.Empty;

    public string PaymentUrl
    {
        get => _paymentUrl;
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("URL не может быть пустым.");
            _paymentUrl = value;
        }
    }

    public string Provider { get; set; }
    public int PointsEarned { get; set; }
    public bool IsEncrypted { get; set; } = true;

    public OnlinePayment(int id, string methodName, decimal minAmount,
                         string paymentUrl, string provider = "GenericPay")
        : base(id, methodName, minAmount, "Онлайн-платеж")
    {
        PaymentUrl = paymentUrl;
        Provider = provider;
    }

    public override string ProcessPayment(decimal amount)
    {
        if (!CheckMinimumAmount(amount) && FallbackMethod != null)
            return $"Онлайн-оплата отклонена (min {MinAmount:C}). " +
                   $"Резерв:\n   -> {FallbackMethod.ProcessPayment(amount)}";

        var baseResult = base.ProcessPayment(amount);
        AddPoints(amount);
        return $"{baseResult}\n   URL: {PaymentUrl} | Провайдер: {Provider} | " +
               $"Зашифровано: {Encrypt()} | Бонусов: {PointsEarned}";
    }

    public override string GetPaymentDetails()
        => base.GetPaymentDetails() + $" | URL: {PaymentUrl} | Провайдер: {Provider}";

    public string Encrypt() => IsEncrypted ? "AES-256" : "нет";

    public void AddPoints(decimal amount) => PointsEarned += (int)(amount / 100m);

    public string Refund(decimal amount) => $"Возврат {amount:C} на карту через {Provider} выполнен.";
}

// СЛОЖНОЕ НАСЛЕДОВАНИЕ: CardPayment : OnlinePayment : PaymentMethod
public class CardPayment : OnlinePayment
{
    public string CardMask { get; set; }
    public string CardHolder { get; set; }
    public bool Is3DSecure { get; set; }

    public CardPayment(int id, string methodName, decimal minAmount,
                       string paymentUrl, string cardMask,
                       string cardHolder, bool is3DSecure = true)
        : base(id, methodName, minAmount, paymentUrl, "CardGateway")
    {
        CardMask = cardMask;
        CardHolder = cardHolder;
        Is3DSecure = is3DSecure;
    }

    public override string ProcessPayment(decimal amount)
    {
        var baseResult = base.ProcessPayment(amount);
        return $"{baseResult}\n   Карта: {CardMask} | Держатель: {CardHolder} | 3D-Secure: {Is3DSecure}";
    }

    public override string GetPaymentDetails()
        => base.GetPaymentDetails() + $" | Карта: {CardMask} | 3DS: {Is3DSecure}";
}

// БАНКОВСКИЙ ПЕРЕВОД
public class BankTransfer : PaymentMethod, IRefundable
{
    public string BankData { get; set; }
    public decimal CommissionPercent { get; set; }
    public string BankName { get; set; }
    public string SwiftCode { get; set; }

    public BankTransfer(int id, string methodName, decimal minAmount,
                        string bankData, decimal commissionPercent = 1.5m,
                        string bankName = "Unknown Bank", string swift = "")
        : base(id, methodName, minAmount, "Банковский перевод")
    {
        BankData = bankData;
        CommissionPercent = commissionPercent;
        BankName = bankName;
        SwiftCode = swift;
    }

    public override bool CheckMinimumAmount(decimal amount)
    {
        decimal commission = amount * CommissionPercent / 100m;
        return (amount + commission) >= MinAmount;
    }

    public override string ProcessPayment(decimal amount)
    {
        if (!CheckMinimumAmount(amount) && FallbackMethod != null)
            return $"Перевод с комиссией {CommissionPercent}% < min ({MinAmount:C}). " +
                   $"Резерв:\n   -> {FallbackMethod.ProcessPayment(amount)}";

        decimal commission = amount * CommissionPercent / 100m;
        return $"Банковский перевод {amount:C} (комиссия {commission:C}) через {BankName}.\n" +
               $"   Реквизиты: {BankData} | SWIFT: {SwiftCode}";
    }

    public override string GetPaymentDetails()
        => base.GetPaymentDetails() + $" | Банк: {BankName} | Комиссия: {CommissionPercent}% | SWIFT: {SwiftCode}";

    public string Refund(decimal amount) => $"Возврат {amount:C} на счет {BankData} выполнен.";
}

// НАЛИЧНЫЕ
public class CashPayment : PaymentMethod, IRefundable
{
    public string CashPickupPoint { get; set; }
    public string WorkingHours { get; set; }
    public string OperatorName { get; set; }

    public CashPayment(int id, string methodName, decimal minAmount,
                       string cashPickupPoint, string workingHours = "09:00-18:00",
                       string operatorName = "Кассир")
        : base(id, methodName, minAmount, "Наличные")
    {
        CashPickupPoint = cashPickupPoint;
        WorkingHours = workingHours;
        OperatorName = operatorName;
    }

    public override string GetPaymentDetails()
        => base.GetPaymentDetails() + $" | Точка: {CashPickupPoint} | Часы: {WorkingHours} | Оператор: {OperatorName}";

    public override string ProcessPayment(decimal amount)
    {
        var baseResult = base.ProcessPayment(amount);
        return baseResult + $"\n   Получить наличные: {CashPickupPoint} ({WorkingHours}), оператор: {OperatorName}";
    }

    public string Refund(decimal amount) => $"Возврат наличных {amount:C} в пункте {CashPickupPoint}.";
}

// ЗАКАЗ
public class Order
{
    public int OrderId { get; set; }
    public decimal TotalAmount { get; set; }
    public IPaymentMethod SelectedPaymentMethod { get; set; }
    public List<string> PaymentHistory { get; } = new List<string>();

    public Order(int orderId, decimal totalAmount, IPaymentMethod method)
    {
        OrderId = orderId;
        TotalAmount = totalAmount;
        SelectedPaymentMethod = method;
    }

    public void PayOrder()
    {
        Console.WriteLine($"[Заказ #{OrderId}] Сумма: {TotalAmount:C}, метод: {SelectedPaymentMethod.MethodName}");
        string status = SelectedPaymentMethod.ProcessPayment(TotalAmount);
        PaymentHistory.Add(status);
        Console.WriteLine($"Результат: {status}\n");
    }
}

// ДЕМОНСТРАЦИЯ
public static class PaymentDemo
{
    public static void Run()
    {
        var cash   = new CashPayment(3, "Наличные", 10m, "г. Москва, ул. Ленина, 10");
        var online = new OnlinePayment(1, "Онлайн-оплата", 500m, "https://pay.example.com/checkout", "YooMoney");
        var bank   = new BankTransfer(2, "Банковский перевод", 1000m, "IBAN: DE12 3456 7890", 2.0m, "Sberbank", "SABRRUMM");
        var card   = new CardPayment(4, "Оплата картой", 300m, "https://card.example.com/pay", "**** **** **** 1234", "IVAN IVANOV");

        online.FallbackMethod = cash;
        bank.FallbackMethod   = cash;
        card.FallbackMethod   = online;

        Console.WriteLine("=== Детали способов оплаты ===\n");
        Console.WriteLine(online.GetPaymentDetails());
        Console.WriteLine(bank.GetPaymentDetails());
        Console.WriteLine(cash.GetPaymentDetails());
        Console.WriteLine(card.GetPaymentDetails());

        new Order(101, 750m, online).PayOrder();
        new Order(102, 100m, online).PayOrder();
        new Order(103, 5000m, bank).PayOrder();
        new Order(104, 400m, card).PayOrder();

        online.MinAmount = 200m;
        online.Deactivate();
        new Order(105, 250m, online).PayOrder();
        online.Activate();
        new Order(106, 250m, online).PayOrder();

        Console.WriteLine(online.Refund(100m));  
        Console.WriteLine(bank.Refund(200m));
        Console.WriteLine(cash.Refund(50m));
        Console.WriteLine($"Бонусы OnlinePayment: {online.PointsEarned}");
    }
}

PaymentDemo.Run();

=== Детали способов оплаты ===

[ID 1] Онлайн-оплата | Мин: ¤500.00 | Валюта: RUB | Активен: True | Онлайн-платеж | URL: https://pay.example.com/checkout | Провайдер: YooMoney
[ID 2] Банковский перевод | Мин: ¤1,000.00 | Валюта: RUB | Активен: True | Банковский перевод | Банк: Sberbank | Комиссия: 2.0% | SWIFT: SABRRUMM
[ID 3] Наличные | Мин: ¤10.00 | Валюта: RUB | Активен: True | Наличные | Точка: г. Москва, ул. Ленина, 10 | Часы: 09:00-18:00 | Оператор: Кассир
[ID 4] Оплата картой | Мин: ¤300.00 | Валюта: RUB | Активен: True | Онлайн-платеж | URL: https://card.example.com/pay | Провайдер: CardGateway | Карта: **** **** **** 1234 | 3DS: True
[Заказ #101] Сумма: ¤750.00, метод: Онлайн-оплата
Результат: Оплата ¤750.00 (RUB) через 'Онлайн-оплата' выполнена.
   URL: https://pay.example.com/checkout | Провайдер: YooMoney | Зашифровано: AES-256 | Бонусов: 7

[Заказ #102] Сумма: ¤100.00, метод: Онлайн-оплата
Результат: Онлайн-оплата отклонена (min ¤500.00). Резерв:
   -> Оплата ¤100.00 (RUB)